<div align="right"><sub>Notebook 最終更新: 2026-03-25 16:59</sub></div>
<h1><strong>05. AIエージェントの基礎</strong></h1>

今回からは，LLMを単体で使うのではなく，役割を持った「エージェント」として組み合わせて，複雑なタスクをこなす方法を学びます．
まずは，回答を行う **Executor** と，その回答をチェックする **Critic** の2役を組み合わせた「自己修正ループ」を体験しましょう．

> **モデルの変更**: エージェントには高い推論・メタ認知能力が求められるため，この回から **Qwen3-8B** を使用します（01〜04回は Qwen2.5-3B-Instruct）．

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes sentence-transformers faiss-cpu peft datasets gradio

import os
import sys
from google.colab import drive

DRIVE_MOUNT_POINT = '/content/drive'
drive.mount(DRIVE_MOUNT_POINT, force_remount=False)

PERSIST_ROOT = os.path.join(DRIVE_MOUNT_POINT, 'MyDrive', 'AIAgent')
PERSIST_INDEX_DIR = os.path.join(PERSIST_ROOT, 'data', 'index')
os.makedirs(PERSIST_INDEX_DIR, exist_ok=True)

REPO_ROOT = '/content/llm_lab'
if not os.path.exists(REPO_ROOT):
    !git clone -b ai_agent https://github.com/akio-kobayashi/llm_lab.git {REPO_ROOT}

os.chdir(REPO_ROOT)
src_path = os.path.abspath('src')
if src_path not in sys.path:
    sys.path.append(src_path)

print('現在の作業ディレクトリ:', os.getcwd())
print('永続ディレクトリ:', PERSIST_ROOT)
from src.common import load_llm, generate_text, AGENT_MODEL_ID
from src.agent_core import LLMExecutorCriticAgent, RoleConfig

model, tokenizer = load_llm(model_id=AGENT_MODEL_ID)
print('準備完了')


## **1. チャット関数の準備**
日本語LLMのチャット機能を使って，システムプロンプトを受け取れるラッパー関数を作成します．

In [ ]:
def llm_chat(system_prompt: str, user_prompt: str, max_tokens: int = 512, temp: float = 0.3):
    return generate_text(model, tokenizer, user_prompt, max_new_tokens=max_tokens, temperature=temp, system_prompt=system_prompt)

---
## **候補A: 論理パズル**
LLMが苦手とする論理的推論タスクで，Executor の初回回答に誤りが含まれ，Critic がそれを検出・修正できるか試します．

In [ ]:
executor_a = RoleConfig(
    name="Executor",
    system_prompt="あなたは論理パズルを解くアシスタントです。必ず日本語で回答してください。ステップごとに推論し、最終的な答えを明示してください。"
)
critic_a = RoleConfig(
    name="Critic",
    system_prompt="あなたは論理パズルの採点者です。必ず日本語で回答してください。回答の推論過程を1ステップずつ検証し、論理的な誤りがないか確認してください。誤りがあれば具体的に指摘し、正しい推論を示してください。問題がなければ「誤りなし」とだけ答えてください。"
)
agent_a = LLMExecutorCriticAgent(llm_chat, role_configs=[executor_a, critic_a])

query_a = """以下の条件から、4人の身長を高い順に並べてください。

・AはCより背が高い
・BはAより背が高い
・DはCより背が高いがAより背が低い
・EはBより背が高い

全員の順序を示し、各ステップの根拠を説明してください。""" #@param{type:'string'}

answer_a, log_a, _ = agent_a.run_pipeline(query_a, max_iterations=2)
print("=== 候補A: 論理パズル ===")
print(log_a)
print("\n=== 最終回答 ===")
print(answer_a)

**正解**: E > B > A > D > C

✅ **確認ポイント**: Executor の初回回答に誤りがあったか？ Critic はそれを見つけたか？ 修正後は正しくなったか？

---

## **候補B: 厳しい制約の文章生成**
制約を厳しくすることで，1回では完璧に書けないタスクにします．

In [ ]:
executor_b = RoleConfig(
    name="Executor",
    system_prompt="あなたは文章作成者です。必ず日本語で回答してください。指定された条件をすべて満たす文章のみを出力してください。余計な説明は不要です。"
)
critic_b = RoleConfig(
    name="Critic",
    system_prompt="あなたは厳格な校閲者です。必ず日本語で回答してください。以下の7項目を1つずつチェックし、✓ か ✗ を付けてください。\n(1) 日時あり\n(2) 場所あり\n(3) 参加条件あり\n(4) 持ち物の指示あり\n(5) です・ます調\n(6) 50字以内（正確に文字数を数えること）\n(7) 元の情報の追加・削除なし\nすべて ✓ なら「誤りなし」、✗ があれば修正案を提示。"
)
agent_b = LLMExecutorCriticAgent(llm_chat, role_configs=[executor_b, critic_b])

query_b = """次の案内文を、以下の7条件をすべて満たすように書き直してください。
(1) 日時  (2) 場所  (3) 参加条件  (4) 持ち物  (5) です・ます調  (6) 50字以内  (7) 情報の追加・削除なし

案内文: 研究室見学をする予定です。来たい人は参加できますが、申込みした人を優先します。今週土曜日の午後2時からで、場所は情報学部1号館3階の301室です。ノートPCを持参してください。""" #@param{type:'string'}

answer_b, log_b, _ = agent_b.run_pipeline(query_b, max_iterations=2)
print("=== 候補B: 厳しい制約 ===")
print(log_b)
print("\n=== 最終回答 ===")
print(answer_b)
print(f"文字数: {len(answer_b)}字")

✅ **確認ポイント**: 50字以内に全情報を収めるのは非常に困難です．Critic は文字数を正確にカウントできたか？ 情報の欠落を指摘できたか？

---

## **候補C: コードのデバッグ**
バグ入りの Python コードを渡し，Executor が修正 → Critic がレビューします．

In [ ]:
executor_c = RoleConfig(
    name="Executor",
    system_prompt="あなたはPythonプログラマです。必ず日本語で説明してください。バグを見つけて修正し、修正後のコード全体と修正理由を出力してください。"
)
critic_c = RoleConfig(
    name="Critic",
    system_prompt="あなたはコードレビュアーです。必ず日本語で回答してください。修正されたコードが正しく動作するか、テストケースを使って検証してください。具体的に入力と期待出力を示し、コードがそれを満たすか確認してください。問題がなければ「誤りなし」とだけ答えてください。"
)
agent_c = LLMExecutorCriticAgent(llm_chat, role_configs=[executor_c, critic_c])

query_c = """以下のPython関数にはバグが2つあります。見つけて修正してください。

```python
def calculate_average(scores):
    \"\"\"成績のリストから平均点を計算する。
    空リストの場合は0を返す。
    100点を超える値は除外する。
    \"\"\"
    total = 0
    for score in scores:
        if score <= 100:
            total += score
    return total / len(scores)
```

テストケース:
- calculate_average([80, 90, 70]) → 80.0
- calculate_average([80, 150, 70]) → 75.0  (150は除外)
- calculate_average([]) → 0""" #@param{type:'string'}

answer_c, log_c, _ = agent_c.run_pipeline(query_c, max_iterations=2)
print("=== 候補C: コードデバッグ ===")
print(log_c)
print("\n=== 最終回答 ===")
print(answer_c)

✅ **確認ポイント**: バグは (1) 空リストでのゼロ除算 (2) 除外した値も `len(scores)` に含まれる、の2つ．Executor は両方見つけたか？ Critic はテストケースで検証したか？

---

## **候補D: ハルシネーション検出**
モデルが知らない情報について質問し，Executor が捏造した場合に Critic がそれを検出できるか試します．

In [ ]:
executor_d = RoleConfig(
    name="Executor",
    system_prompt="あなたは知識豊富なアシスタントです。必ず日本語で回答してください。質問に対してできる限り詳しく回答してください。"
)
critic_d = RoleConfig(
    name="Critic",
    system_prompt="あなたは厳格なファクトチェッカーです。必ず日本語で回答してください。回答に含まれる事実関係を1つずつ確認してください。確信が持てない情報や、明らかに捏造の可能性がある具体的な数値・人名・日付がないか厳しくチェックしてください。怪しい箇所があれば『この情報は確認できません』と指摘してください。問題がなければ「誤りなし」とだけ答えてください。"
)
agent_d = LLMExecutorCriticAgent(llm_chat, role_configs=[executor_d, critic_d])

query_d = """2027年に日本で開催予定の「第5回国際AIサミット」について、開催都市、主催者、主要な議題を教えてください。""" #@param{type:'string'}

answer_d, log_d, _ = agent_d.run_pipeline(query_d, max_iterations=2)
print("=== 候補D: ハルシネーション検出 ===")
print(log_d)
print("\n=== 最終回答 ===")
print(answer_d)

✅ **確認ポイント**: 「第5回国際AIサミット」は架空のイベントです．Executor が具体的な情報を捏造した場合，Critic はそれを「確認できない」と指摘できたか？

---

## **結果の比較**

| 候補 | タスク | Executor の初回エラー | Critic の検出 | 修正後の改善 | 評価 |
|:--|:--|:--|:--|:--|:--|
| A | 論理パズル | | | | |
| B | 厳しい制約(50字) | | | | |
| C | コードデバッグ | | | | |
| D | ハルシネーション検出 | | | | |

**Executor の初回回答に誤りがあり，Critic がそれを検出し，修正後に改善が見られる** 候補を，本番のタスクとして採用します．

## **まとめ**
- 1つのプロンプトで完璧な回答を求める（Zero-shot）よりも，役割を分けて「自分で自分のミスを直す」プロセスを入れることで，より信頼性の高い回答が得られるようになります．
- **Critic の役割** は「他の回答を客観的に評価する」というメタ認知的なタスクであり，Executor より高い推論能力が求められます．そのため，エージェントパターンには一定以上のモデル規模が必要です．
- **プロンプトの設計** も重要です．チェックリスト形式のプロンプトを Critic に与えることで，的確で構造化された評価が可能になります．